# Hierarchy-saliency experiments

This notebook explores edge maps induced by morphological trees. A formal map is `Phi(H)` for a connected hierarchy with a compatible valuation: transition edges receive the valuation of `LCA(owner(p), owner(q))`, while same-owner edges receive zero.

Pixel images are C++ display projections of incident edge values. The canonical representation remains the `sources`, `targets`, and `values` dictionary. Primary reference: Jean Cousty, Laurent Najman, Yukiko Kenmochi, and Silvio Guimarães, [*Hierarchical segmentations with graphs: quasi-flat zones, minimum spanning trees, and saliency maps*](https://doi.org/10.1007/s10851-017-0768-7), *Journal of Mathematical Imaging and Vision* 60(4), 479–502, 2018. See the [code-to-paper correspondence](../docs/saliency.md#primary-references-and-implementation-correspondence).

## Evaluated policies

- `computeSaliencyEdgeMap`: formal LCA projection of a compatible hierarchy valuation.
- `validateHierarchyValuation`: checks finiteness, rootward monotonicity, and optional non-negativity.
- `rankHierarchyValuation`: creates dense node levels while preserving collapses.
- `computeTopologicalLevelEdgeMap`: uses strict structural levels.
- `computeNormalizedAltitudeEdgeMap`: orients max/min altitude to increase toward the root and normalizes to `[0, 1]`.
- `computeFormalSaliencyEdgeMap`: builds the Cousty persistence MST/BPTAO map.
- `computeMonotoneExtinctionProjection`: projects the former max-descendant extinction valuation by LCA.
- `thresholdCut` and `nodeContourEdges`: materialize cuts or assign transition edges to their LCA node.

In [ ]:
from __future__ import annotations

import mmcfilters
import matplotlib.pyplot as plt
import numpy as np
from skimage import data

%matplotlib inline
print(f"mmcfilters: {mmcfilters.__version__}")


In [ ]:
def read_uint8_image(path: pathlib.Path, crop: tuple[slice, slice] | None = None) -> np.ndarray:
    image = plt.imread(path)
    if image.ndim == 3:
        image = image[..., 0]
    if image.dtype != np.uint8:
        image = np.asarray(image, dtype=np.float32)
        image = image - np.nanmin(image)
        max_value = np.nanmax(image)
        if max_value > 0:
            image = image / max_value
        image = np.round(255 * image).astype(np.uint8)
    if crop is not None:
        image = image[crop]
    return np.ascontiguousarray(image)


def edge_map_to_pixel_image(edge_map: dict, reducer: str = "max") -> np.ndarray:
    if reducer == "max":
        cpp_reducer = mmcfilters.EdgeToPixelReducer.Max
    elif reducer == "mean":
        cpp_reducer = mmcfilters.EdgeToPixelReducer.Mean
    else:
        raise ValueError("reducer must be 'max' or 'mean'")
    return mmcfilters.HierarchySaliencyMapProjection.edgeMapToPixelImage(edge_map, cpp_reducer)


def summarize_edge_map(name: str, edge_map: dict) -> None:
    values = np.asarray(edge_map["values"])
    print(
        f"{name:28s} edges={values.size:7d} "
        f"dtype={values.dtype!s:8s} min={values.min():10.4g} "
        f"max={values.max():10.4g} mean={values.mean():10.4g}"
    )


def plot_edge_maps(image: np.ndarray, maps: list[tuple[str, dict]], reducer: str = "max", cmap: str = "gray") -> None:
    fig, axes = plt.subplots(1, len(maps) + 1, figsize=(4 * (len(maps) + 1), 4), constrained_layout=True)
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("image")
    axes[0].axis("off")
    for ax, (title, edge_map) in zip(axes[1:], maps):
        vis = edge_map_to_pixel_image(edge_map, reducer=reducer)
        im = ax.imshow(vis, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()


def top_edges(edge_map: dict, k: int = 10) -> list[tuple[int, int, float]]:
    sources = np.asarray(edge_map["sources"], dtype=np.int64)
    targets = np.asarray(edge_map["targets"], dtype=np.int64)
    values = np.asarray(edge_map["values"], dtype=np.float64)
    order = np.argsort(values)[::-1][:k]
    return [(int(sources[i]), int(targets[i]), float(values[i])) for i in order]


def normalize_values(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return values.copy()
    vmin = float(np.nanmin(values))
    vmax = float(np.nanmax(values))
    if vmax <= vmin:
        return np.zeros_like(values, dtype=np.float64)
    return (values - vmin) / (vmax - vmin)


def edge_gradient_values(image_uint8: np.ndarray, edge_map: dict) -> np.ndarray:
    flat = np.asarray(image_uint8, dtype=np.float64).reshape(-1)
    sources = np.asarray(edge_map["sources"], dtype=np.int64)
    targets = np.asarray(edge_map["targets"], dtype=np.int64)
    return np.abs(flat[sources] - flat[targets])


def edge_map_with_values(edge_map: dict, values: np.ndarray) -> dict:
    out = dict(edge_map)
    out["values"] = np.asarray(values)
    return out


def contour_cut_to_edge_map(contour: dict, value: float = 1.0) -> dict:
    sources = np.asarray(contour["sources"], dtype=np.int64)
    out = dict(contour)
    out["values"] = np.full(sources.shape[0], value, dtype=np.float64)
    return out


def normalized_max_tree_node_scores(tree) -> np.ndarray:
    alive = np.asarray(tree.aliveNodeIds, dtype=np.int64)
    scores = np.zeros(int(tree.numInternalNodeSlots), dtype=np.float64)
    if alive.size == 0:
        return scores
    altitudes = np.zeros_like(scores)
    for node_id in alive:
        altitudes[node_id] = float(tree.getAltitude(int(node_id)))
    live_altitudes = altitudes[alive]
    amin = float(live_altitudes.min())
    amax = float(live_altitudes.max())
    if amax <= amin:
        return scores
    scores[alive] = (amax - altitudes[alive]) / (amax - amin)
    return scores


def incremental_contour_node_summary(contours: dict, node_scores: np.ndarray, threshold: float, top: int = 8) -> list[tuple[int, int, float]]:
    offsets = np.asarray(contours["offsets"], dtype=np.int64)
    counts = offsets[1:] - offsets[:-1]
    scores = np.asarray(node_scores, dtype=np.float64)
    selected = np.flatnonzero((counts > 0) & (scores >= threshold))
    if selected.size == 0:
        return []
    order = selected[np.argsort(counts[selected])[::-1][:top]]
    return [(int(node), int(counts[node]), float(scores[node])) for node in order]


def gradient_weighted_edge_map(image_uint8: np.ndarray, edge_map: dict, mode: str = "multiply", alpha: float = 0.5) -> tuple[dict, dict]:
    saliency = normalize_values(np.asarray(edge_map["values"], dtype=np.float64))
    gradient = normalize_values(edge_gradient_values(image_uint8, edge_map))
    if mode == "multiply":
        combined = saliency * gradient
    elif mode == "blend":
        combined = alpha * saliency + (1.0 - alpha) * gradient
    else:
        raise ValueError("mode must be 'multiply' or 'blend'")
    return edge_map_with_values(edge_map, combined), edge_map_with_values(edge_map, gradient)


In [ ]:
CROP = None  # Example: (slice(64, 224), slice(64, 224))
RADIUS = 1.5

image = np.ascontiguousarray(data.camera(), dtype=np.uint8)
if CROP is not None:
    image = np.ascontiguousarray(image[CROP])
print(image.shape, image.dtype, int(image.min()), int(image.max()))

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray")
plt.title("skimage.data.camera()")
plt.axis("off");


In [ ]:
max_tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(image, radius=RADIUS)
min_tree = mmcfilters.MorphologicalTreeFactory.createMinTree(image, radius=RADIUS)

print("max-tree nodes:", max_tree.numInternalNodeSlots, "alive:", len(max_tree.aliveNodeIds))
print("min-tree nodes:", min_tree.numInternalNodeSlots, "alive:", len(min_tree.aliveNodeIds))

In [ ]:
max_level = mmcfilters.HierarchySaliencyMap.computeTopologicalLevelEdgeMap(max_tree)
max_normalized = mmcfilters.HierarchySaliencyMap.computeNormalizedAltitudeEdgeMap(max_tree)

area = mmcfilters.Attribute.computeSingleTopologyAttribute(max_tree, mmcfilters.Attribute.AREA).astype(np.float32)
mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(max_tree, area, nonnegative=True)
area_ranked = mmcfilters.HierarchySaliencyMapValidation.rankHierarchyValuation(max_tree, area)

max_area = mmcfilters.HierarchySaliencyMap.computeSaliencyEdgeMap(max_tree, area)
max_area_ranked = mmcfilters.HierarchySaliencyMap.computeSaliencyEdgeMap(max_tree, area_ranked)

extinction = mmcfilters.ExtinctionValues(max_tree, area)
extinction_valuation = extinction.getExtinctionValueAttribute()
extinction_ranked_valuation = extinction.computeRankedExtinctionValueAttribute()
mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(
    max_tree, extinction_ranked_valuation, nonnegative=True
)

max_extinction_persistence = extinction.computeFormalSaliencyEdgeMap(ranked=True)
max_extinction_monotone = extinction.computeMonotoneExtinctionProjection(ranked=True)
persistence_equals_monotone = np.array_equal(
    max_extinction_persistence["values"], max_extinction_monotone["values"]
)
print("persistence equals monotone LCA projection:", persistence_equals_monotone)
assert not persistence_equals_monotone

min_normalized = mmcfilters.HierarchySaliencyMap.computeNormalizedAltitudeEdgeMap(min_tree)

edge_maps = {
    "max-tree topological level": max_level,
    "max-tree normalized altitude": max_normalized,
    "max-tree formal area": max_area,
    "max-tree ranked area": max_area_ranked,
    "max-tree extinction persistence": max_extinction_persistence,
    "max-tree monotone extinction LCA": max_extinction_monotone,
    "min-tree normalized altitude": min_normalized,
}

for name, edge_map in edge_maps.items():
    summarize_edge_map(name, edge_map)


In [ ]:
plot_edge_maps(
    image,
    [
        ("topological level", max_level),
        ("normalized altitude", max_normalized),
        ("formal area", max_area),
        ("ranked extinction persistence", max_extinction_persistence),
    ],
    reducer="max",
)


In [ ]:
plot_edge_maps(
    image,
    [
        ("max-tree normalized", max_normalized),
        ("min-tree normalized", min_normalized),
    ],
    reducer="max",
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), constrained_layout=True)

for ax, (title, edge_map) in zip(
    axes,
    [
        ("topological level", max_level),
        ("normalized altitude", max_normalized),
        ("formal area", max_area),
        ("ranked extinction persistence", max_extinction_persistence),
    ],
):
    values = np.asarray(edge_map["values"], dtype=np.float64)
    ax.hist(values, bins=64, color="0.25")
    ax.set_title(title)
    ax.set_xlabel("edge value")
    ax.set_ylabel("frequency")

plt.show()


In [ ]:
for name in ["max-tree topological level", "max-tree normalized altitude", "max-tree formal area", "max-tree extinction persistence"]:
    print("\n", name)
    for source, target, value in top_edges(edge_maps[name], k=8):
        sr, sc = divmod(source, image.shape[1])
        tr, tc = divmod(target, image.shape[1])
        print(f"  ({sr:3d},{sc:3d}) -> ({tr:3d},{tc:3d})  value={value:.6g}")


## Before/after area filtering

The next cell applies area threshold 100 to a max-tree, rebuilds the filtered hierarchy, and compares normalized-altitude saliency before and after. Pixel images aggregate incident edge values by maximum.

In [ ]:
AREA_THRESHOLD = 5000.0

def saliency_map_for_image(image_uint8: np.ndarray, radius: float = RADIUS, policy: str = "normalized_altitude"):
    tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(np.ascontiguousarray(image_uint8), radius=radius)
    if policy == "normalized_altitude":
        edge_map = mmcfilters.HierarchySaliencyMap.computeNormalizedAltitudeEdgeMap(tree)
    elif policy == "topological_level":
        edge_map = mmcfilters.HierarchySaliencyMap.computeTopologicalLevelEdgeMap(tree)
    else:
        raise ValueError("policy must be 'normalized_altitude' or 'topological_level'")
    return tree, edge_map, edge_map_to_pixel_image(edge_map, reducer="max")

def plot_saliency_filter_comparison(original_image, filtered_image, before_saliency, after_saliency, threshold):
    diff = after_saliency - before_saliency
    saliency_vmax = max(float(before_saliency.max()), float(after_saliency.max()), 1e-12)
    diff_abs = max(abs(float(diff.min())), abs(float(diff.max())), 1e-12)

    fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
    axes[0, 0].imshow(original_image, cmap="gray")
    axes[0, 0].set_title("original image")
    axes[0, 1].imshow(filtered_image, cmap="gray")
    axes[0, 1].set_title(f"area filter > {threshold:g}")
    axes[0, 2].imshow(np.abs(filtered_image.astype(np.int16) - original_image.astype(np.int16)), cmap="gray")
    axes[0, 2].set_title("absolute change")

    im_before = axes[1, 0].imshow(before_saliency, cmap="gray", vmin=0, vmax=saliency_vmax)
    axes[1, 0].set_title("saliency before")
    im_after = axes[1, 1].imshow(after_saliency, cmap="gray", vmin=0, vmax=saliency_vmax)
    axes[1, 1].set_title("saliency after")
    im_diff = axes[1, 2].imshow(diff, cmap="coolwarm", vmin=-diff_abs, vmax=diff_abs)
    axes[1, 2].set_title("after - before")

    for ax in axes.ravel():
        ax.axis("off")
    fig.colorbar(im_before, ax=axes[1, 0], fraction=0.046, pad=0.04)
    fig.colorbar(im_after, ax=axes[1, 1], fraction=0.046, pad=0.04)
    fig.colorbar(im_diff, ax=axes[1, 2], fraction=0.046, pad=0.04)
    plt.show()

area_for_filter = mmcfilters.Attribute.computeSingleTopologyAttribute(max_tree, mmcfilters.Attribute.AREA).astype(np.float32)
filtered_area_100 = mmcfilters.AttributeFilters(max_tree).filteringByPruningMin(area_for_filter, AREA_THRESHOLD)
filtered_area_100 = np.ascontiguousarray(filtered_area_100)

before_tree, before_edge_map, before_saliency = saliency_map_for_image(image, policy="normalized_altitude")
after_tree, after_edge_map, after_saliency = saliency_map_for_image(filtered_area_100, policy="normalized_altitude")

print("area threshold:", AREA_THRESHOLD)
print("pixels changed:", int(np.count_nonzero(filtered_area_100 != image)))
summarize_edge_map("saliency before filtering", before_edge_map)
summarize_edge_map("saliency after filtering", after_edge_map)

plot_saliency_filter_comparison(image, filtered_area_100, before_saliency, after_saliency, AREA_THRESHOLD)


## Gradient-weighted display

The gradient is an external display weight. Every edge receives normalized `|I[p] - I[q]|`, multiplied by normalized hierarchy saliency. This emphasizes boundaries strong in both hierarchy and local contrast, but is not the canonical hierarchy saliency map.

In [ ]:
before_weighted_gradient, before_gradient_edge = gradient_weighted_edge_map(image, before_edge_map, mode="multiply")
after_weighted_gradient, after_gradient_edge = gradient_weighted_edge_map(filtered_area_100, after_edge_map, mode="multiply")

summarize_edge_map("gradient before", before_gradient_edge)
summarize_edge_map("saliency*gradient before", before_weighted_gradient)
summarize_edge_map("gradient after", after_gradient_edge)
summarize_edge_map("saliency*gradient after", after_weighted_gradient)

before_saliency_vis = edge_map_to_pixel_image(before_edge_map, reducer="max")
after_saliency_vis = edge_map_to_pixel_image(after_edge_map, reducer="max")
before_gradient_vis = edge_map_to_pixel_image(before_gradient_edge, reducer="max")
after_gradient_vis = edge_map_to_pixel_image(after_gradient_edge, reducer="max")
before_weighted_vis = edge_map_to_pixel_image(before_weighted_gradient, reducer="max")
after_weighted_vis = edge_map_to_pixel_image(after_weighted_gradient, reducer="max")

fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
rows = [
    ("before", image, before_gradient_vis, before_saliency_vis, before_weighted_vis),
    ("after", filtered_area_100, after_gradient_vis, after_saliency_vis, after_weighted_vis),
]
for row_idx, (label, shown_image, gradient_vis, saliency_vis, weighted_vis) in enumerate(rows):
    axes[row_idx, 0].imshow(shown_image, cmap="gray")
    axes[row_idx, 0].set_title(f"image {label}")
    im_grad = axes[row_idx, 1].imshow(gradient_vis, cmap="gray", vmin=0, vmax=1)
    axes[row_idx, 1].set_title(f"gradient {label}")
    im_sal = axes[row_idx, 2].imshow(saliency_vis, cmap="gray", vmin=0, vmax=max(before_saliency_vis.max(), after_saliency_vis.max()))
    axes[row_idx, 2].set_title(f"saliency {label}")
    im_weighted = axes[row_idx, 3].imshow(weighted_vis, cmap="gray", vmin=0, vmax=1)
    axes[row_idx, 3].set_title(f"saliency * gradient {label}")
    for col in range(4):
        axes[row_idx, col].axis("off")
    fig.colorbar(im_grad, ax=axes[row_idx, 1], fraction=0.046, pad=0.04)
    fig.colorbar(im_sal, ax=axes[row_idx, 2], fraction=0.046, pad=0.04)
    fig.colorbar(im_weighted, ax=axes[row_idx, 3], fraction=0.046, pad=0.04)
plt.show()

weighted_diff = after_weighted_vis - before_weighted_vis
diff_abs = max(abs(float(weighted_diff.min())), abs(float(weighted_diff.max())), 1e-12)
plt.figure(figsize=(5, 4))
plt.imshow(weighted_diff, cmap="coolwarm", vmin=-diff_abs, vmax=diff_abs)
plt.title("weighted saliency: after - before")
plt.axis("off")
plt.colorbar(fraction=0.046, pad=0.04)
plt.show()


## Incremental contours projected from saliency

A level cut selects transition edges of nodes with score at least the threshold. `computeIncrementalNodeContours` stores the edge slice `offsets[u]:offsets[u + 1]` for each LCA node. Same-owner zero edges remain implicit.

`projectNodeValuation` and `thresholdByNodeValuation` operate on this sparse transition support and do not validate valuations, so the cell validates `max_node_scores` before projection.

In [ ]:
CONTOUR_THRESHOLDS = [0.25, 0.50, 0.75]

incremental_contours = mmcfilters.HierarchySaliencyMapProjection.computeIncrementalNodeContours(max_tree)
max_node_scores = normalized_max_tree_node_scores(max_tree)
mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(max_tree, max_node_scores, nonnegative=True)
projected_from_nodes = mmcfilters.HierarchySaliencyMapProjection.projectNodeValuation(
    incremental_contours,
    max_node_scores,
)

projected_vis = edge_map_to_pixel_image(projected_from_nodes, reducer="max")
reference_vis = edge_map_to_pixel_image(max_normalized, reducer="max")
print("projected transition edges:", len(projected_from_nodes["sources"]))
print("edges in complete formal map:", len(max_normalized["sources"]))

contour_maps = []
for threshold in CONTOUR_THRESHOLDS:
    cut = mmcfilters.HierarchySaliencyMapProjection.thresholdByNodeValuation(
        incremental_contours,
        max_node_scores,
        threshold,
    )
    contour_maps.append((f"lambda >= {threshold:.2f}", contour_cut_to_edge_map(cut)))
    fraction = len(cut["sources"]) / max(1, len(projected_from_nodes["sources"]))
    print(f"lambda={threshold:.2f}: {len(cut['sources']):6d} edges ({fraction:6.2%})")
    print("  most frequent nodes:", incremental_contour_node_summary(incremental_contours, max_node_scores, threshold, top=6))

plot_edge_maps(image, contour_maps, reducer="max", cmap="gray")


## Further experiments

- Compare radii 1.0 and 1.5.
- Replace `AREA` with another increasing attribute and validate it explicitly.
- Compare raw node valuations, dense node ranks, and canonical edge ranks.
- Compare persistence saliency with the explicit monotone extinction projection.
- Compare formal edge cuts with node-attribute contour cuts.
- Try blend rather than multiplication for external edge weights.